In [ ]:
!pip install torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 37.8 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# import
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import numpy as numpy
import os
from PIL import Image

from tqdm import tqdm
import wandb
wandb.login()


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: km936 (km936-cornell-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

Load the Dataset

In [ ]:
!wget -c http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_HR.zip

--2026-05-11 02:52:53--  http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_HR.zip
Resolving data.vision.ee.ethz.ch (data.vision.ee.ethz.ch)... 129.132.52.178, 2001:67c:10ec:36c2::178
Connecting to data.vision.ee.ethz.ch (data.vision.ee.ethz.ch)|129.132.52.178|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_HR.zip [following]
--2026-05-11 02:52:54--  https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_HR.zip
Connecting to data.vision.ee.ethz.ch (data.vision.ee.ethz.ch)|129.132.52.178|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3530603713 (3.3G) [application/zip]
Saving to: ‘DIV2K_train_HR.zip’

DIV2K_train_HR.zip  100%[===================>]   3.29G  19.5MB/s    in 2m 52s  

2026-05-11 02:55:46 (19.6 MB/s) - ‘DIV2K_train_HR.zip’ saved [3530603713/3530603713]



In [ ]:
!unzip DIV2K_train_HR.zip

Archive:  DIV2K_train_HR.zip
   creating: DIV2K_train_HR/
  inflating: DIV2K_train_HR/0103.png  
  inflating: DIV2K_train_HR/0413.png  
  inflating: DIV2K_train_HR/0031.png  
  inflating: DIV2K_train_HR/0660.png  
  inflating: DIV2K_train_HR/0126.png  
  inflating: DIV2K_train_HR/0793.png  
  inflating: DIV2K_train_HR/0764.png  
  inflating: DIV2K_train_HR/0550.png  
  inflating: DIV2K_train_HR/0437.png  
  inflating: DIV2K_train_HR/0374.png  
  inflating: DIV2K_train_HR/0755.png  
  inflating: DIV2K_train_HR/0614.png  
  inflating: DIV2K_train_HR/0646.png  
  inflating: DIV2K_train_HR/0371.png  
  inflating: DIV2K_train_HR/0312.png  
  inflating: DIV2K_train_HR/0108.png  
  inflating: DIV2K_train_HR/0556.png  
  inflating: DIV2K_train_HR/0794.png  
  inflating: DIV2K_train_HR/0722.png  
  inflating: DIV2K_train_HR/0780.png  
  inflating: DIV2K_train_HR/0555.png  
  inflating: DIV2K_train_HR/0439.png  
  inflating: DIV2K_train_HR/0396.png  
  inflating: DIV2K_train_HR/0666.png  
  infl

In [ ]:
!rm -r DIV2K_train_HR.zip

In [ ]:
!wget -c http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip

--2026-05-11 02:56:28--  http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip
Resolving data.vision.ee.ethz.ch (data.vision.ee.ethz.ch)... 129.132.52.178, 2001:67c:10ec:36c2::178
Connecting to data.vision.ee.ethz.ch (data.vision.ee.ethz.ch)|129.132.52.178|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip [following]
--2026-05-11 02:56:29--  https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip
Connecting to data.vision.ee.ethz.ch (data.vision.ee.ethz.ch)|129.132.52.178|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 448993893 (428M) [application/zip]
Saving to: ‘DIV2K_valid_HR.zip’

DIV2K_valid_HR.zip  100%[===================>] 428.19M  20.2MB/s    in 23s     

2026-05-11 02:56:52 (19.0 MB/s) - ‘DIV2K_valid_HR.zip’ saved [448993893/448993893]



In [ ]:
!unzip DIV2K_valid_HR.zip

Archive:  DIV2K_valid_HR.zip
   creating: DIV2K_valid_HR/
  inflating: DIV2K_valid_HR/0897.png  
  inflating: DIV2K_valid_HR/0887.png  
  inflating: DIV2K_valid_HR/0806.png  
  inflating: DIV2K_valid_HR/0834.png  
  inflating: DIV2K_valid_HR/0896.png  
  inflating: DIV2K_valid_HR/0881.png  
  inflating: DIV2K_valid_HR/0828.png  
  inflating: DIV2K_valid_HR/0833.png  
  inflating: DIV2K_valid_HR/0877.png  
  inflating: DIV2K_valid_HR/0826.png  
  inflating: DIV2K_valid_HR/0879.png  
  inflating: DIV2K_valid_HR/0812.png  
  inflating: DIV2K_valid_HR/0809.png  
  inflating: DIV2K_valid_HR/0865.png  
  inflating: DIV2K_valid_HR/0882.png  
  inflating: DIV2K_valid_HR/0830.png  
  inflating: DIV2K_valid_HR/0892.png  
  inflating: DIV2K_valid_HR/0859.png  
  inflating: DIV2K_valid_HR/0858.png  
  inflating: DIV2K_valid_HR/0816.png  
  inflating: DIV2K_valid_HR/0836.png  
  inflating: DIV2K_valid_HR/0857.png  
  inflating: DIV2K_valid_HR/0824.png  
  inflating: DIV2K_valid_HR/0823.png  
  infl

In [ ]:
!rm -r DIV2K_valid_HR.zip

Dataset Settings

In [ ]:
# check where dataset is loaded relative to colab files
root_path_to_image_train = '/content/DIV2K_train_HR'
root_path_to_image_valid = '/content/DIV2K_valid_HR'

# set the batch size and workers here
batch_size = 4
num_workers = 0

In [ ]:
# Set up dataset

class Div2KDataset(Dataset):
    def __init__(self, root, transforms=None):
        self.path_to_img = []
        for f in os.listdir(root):
            if f.endswith('png'):
                self.path_to_img.append(os.path.join(root, f))

        self.transforms = transforms

    def __len__(self):
        return len(self.path_to_img)

    def __getitem__(self, idx):
        image = Image.open(self.path_to_img[idx]).convert('RGB')
        if self.transforms:
            image = self.transforms(image)
        return image

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(360, pad_if_needed=True),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])
#converts each pixel from [0,1] to [-1,1]

validation_transform = transforms.Compose([
    transforms.CenterCrop(360), # keep crop deterministic
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]) # same normalization as train
])

train_dataset = Div2KDataset(root_path_to_image_train, transforms=train_transform)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)

valid_dataset = Div2KDataset(root_path_to_image_valid, transforms=validation_transform)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

In [ ]:
# Check dataset sizes
print(f"Train samples: {len(train_dataset)}")   # 800
print(f"Valid samples: {len(valid_dataset)}")   # 100

# Check a single image loading correctly
sample = train_dataset[0]
print(f"Sample type: {type(sample)}")
print(f"Sample shape: {sample.shape}") # [3, 360, 360]

# Check dataloaders
train_batch = next(iter(train_loader))
print(f"Train batch shape: {train_batch.shape}")  # [batch_size, 3, 360, 360]

valid_batch = next(iter(valid_loader))
print(f"Valid batch shape: {valid_batch.shape}") # [batch_size, 3, 360, 360]

Train samples: 800
Valid samples: 100
Sample type: <class 'torch.Tensor'>
Sample shape: torch.Size([3, 360, 360])
Train batch shape: torch.Size([4, 3, 360, 360])
Valid batch shape: torch.Size([4, 3, 360, 360])


Evaluation Functions

In [ ]:
from torchmetrics.image import StructuralSimilarityIndexMeasure

def psnr(cover_image, stego_image):
  """
  Inputs:
  - cover_image: N x 3 x H x W (normalized to [-1, 1])
  - stego_image: N x 3 x H x W (normalized to [-1, 1])

  Returns: scalar average PSNR in dB across the batch
  """
  # pixel range is 2.0 for images in [-1, 1]
  max_val = 2.0
  mse = F.mse_loss(stego_image, cover_image)
  return (10 * torch.log10(max_val ** 2 / mse)).item()

def ssim(cover_image, stego_image):
  """
  Inputs:
  - cover_image: N x 3 x H x W (normalized to [-1, 1])
  - stego_image: N x 3 x H x W (normalized to [-1, 1])

  Returns: scalar SSIM value, closer to 1 is better
  """
  device = cover_image.device
  metric = StructuralSimilarityIndexMeasure(data_range=2.0).to(device)
  return metric(stego_image, cover_image).item()

def evaluate(encoder, decoder, dataloader, D, device="cpu"):
  """
  Inputs:
  - encoder: trained encoder model
  - decoder: trained decoder model
  - dataloader: DataLoader over the validation set
  - D: bits per pixel (must match what encoder/decoder were trained with)
  - device: "cpu" or "cuda"

  Returns: dict with RS-BPP (aggregated over the whole val set) and
           image-weighted average PSNR / SSIM.

  Notes:
  - RS-BPP is computed by summing wrong-bit counts across the entire val set
    and applying the formula once. Per-batch averaging would inflate it
    because of the max(0, ·) clamp (non-linear → mean(max(...)) ≠ max(mean(...))).
  - PSNR / SSIM are weighted by batch size N so the smaller final batch
    (e.g. 4 images out of 100) doesn't get over-weighted.
  """

  #Switch the models from training mode to evaluation mode.
  #BatchNorm behaves differently during evaluation (uses running mean and variance)
  encoder.eval()
  decoder.eval()

  total_wrong = 0
  total_bits  = 0
  psnr_sum = 0.0
  ssim_sum = 0.0
  total_images = 0

  with torch.no_grad():
    for cover_image in dataloader: #cover_image is shape (N, 3, H, W)
      cover_image = cover_image.to(device)
      N, _, H, W = cover_image.shape

      # generate random binary message for this batch
      message = torch.randint(0, 2, (N, D, H, W), dtype=torch.float, device=device)

      # encode and decode
      stego_image = encoder(cover_image, message)
      decoded_message = decoder(stego_image)

      # accumulate raw bit-error counts so we can apply the RS-BPP formula
      # ONCE at the end (avoids the max(0, ·) clamp bias from per-batch averaging)
      predicted_bits = (decoded_message > 0).float()
      total_wrong += (predicted_bits != message).sum().item()
      total_bits  += message.numel()

      # weight image-level metrics by batch size so the trailing partial batch
      # doesn't get over-weighted
      psnr_sum += psnr(cover_image, stego_image) * N
      ssim_sum += ssim(cover_image, stego_image) * N
      total_images += N

  p = total_wrong / total_bits
  rs_bpp = max(0.0, D * (1 - 2 * p))
  acc = 1.0 - (total_wrong / total_bits)

  return {
    "RS-BPP": rs_bpp,
    "PSNR": psnr_sum / total_images,
    "SSIM": ssim_sum / total_images,
    "Acc": acc,
  }


# ---------- Qualitative logging helpers (used by the Trainer for wandb image panels) ----------

def _denorm(x):
    # undo image normalization so we can visualize them during wandb logging
    return (x * 0.5 + 0.5).clamp(0, 1)

def _qualitative_samples_dict(encoder, decoder, fixed_batch, D, device, max_images=4):
    """
    Builds (but does NOT log) a wandb-ready dict of qualitative panels for a fixed val batch
    so we can see the same images evolve across epochs. The Trainer merges this dict into
    its eval-metrics log call so all per-epoch values share the same wandb _step.

    Panels:
      - cover:     original images
      - stego:     encoder output given a random message
      - residual:  visual differences between original and encoded image. bright spots show differences
    """

    #switch models to eval mode
    encoder.eval()
    decoder.eval()

    with torch.no_grad():
        cover = fixed_batch.to(device)[:max_images]
        N, _, H, W = cover.shape
        # use a fixed seed to generate the message so that it's the same every epoch for this panel
        g = torch.Generator(device=device).manual_seed(0)
        M = torch.randint(0, 2, (N, D, H, W), generator=g, device=device).float()

        stego = encoder(cover, M)

        cover_v = _denorm(cover)
        stego_v = _denorm(stego)
        #The residual image will show bright spots where the encoder modified the cover image
        #Helps see if the embedding is spatially uniform, and if it hides bits in edgy textured areas or smooth areas
        residual_v = (stego_v - cover_v).abs().mul(10).clamp(0, 1)

        return {
            "val/samples/cover":    [wandb.Image(img) for img in cover_v.cpu()],
            "val/samples/stego":    [wandb.Image(img) for img in stego_v.cpu()],
            "val/samples/residual_x10": [wandb.Image(img) for img in residual_v.cpu()],
        }


Encoders

In [ ]:
from torch import nn
import torch.nn.functional as F

class WindowAttention(nn.Module):
    def __init__(self, channels, window_size=8, num_heads=4):
        super().__init__()
        self.window_size = window_size
        self.num_heads = num_heads
        self.head_dim = channels // num_heads
        self.scale = self.head_dim ** -0.5

        self.qkv = nn.Conv2d(channels, channels * 3, 1)
        self.proj = nn.Conv2d(channels, channels, 1)

    def forward(self, x):
        N, C, H, W = x.shape
        ws = self.window_size

        pad_h = (ws - H % ws) % ws
        pad_w = (ws - W % ws) % ws
        x = F.pad(x, (0, pad_w, 0, pad_h))
        _, _, Hp, Wp = x.shape

        x = x.view(N, C, Hp // ws, ws, Wp // ws, ws)
        x = x.permute(0, 2, 4, 1, 3, 5).contiguous()
        x = x.view(-1, C, ws, ws)

        # QKV
        qkv = self.qkv(x)
        qkv = qkv.view(-1, 3, self.num_heads, self.head_dim, ws * ws)
        qkv = qkv.permute(1, 0, 2, 4, 3)
        q, k, v = qkv.unbind(0)

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        out = (attn @ v)

        out = out.permute(0, 1, 3, 2).contiguous()
        out = out.view(-1, C, ws, ws)
        out = self.proj(out)

        out = out.view(N, Hp // ws, Wp // ws, C, ws, ws)
        out = out.permute(0, 3, 1, 4, 2, 5).contiguous()
        out = out.view(N, C, Hp, Wp)

        out = out[:, :, :H, :W]
        return out


class AttentionEncoder(nn.Module):
    def __init__(self, D: int, window_size=8, num_heads=4):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 32, 3, 1, "same"),
            nn.LeakyReLU(inplace=True),
            nn.BatchNorm2d(32)
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(32 + D, 32, 3, 1, "same"),
            nn.LeakyReLU(inplace=True),
            nn.BatchNorm2d(32)
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(32 * 2 + D, 32, 3, 1, "same"),
            nn.LeakyReLU(inplace=True),
            nn.BatchNorm2d(32)
        )

        self.attn = WindowAttention(32, window_size=window_size, num_heads=num_heads)
        self.norm = nn.BatchNorm2d(32)

        self.conv4 = nn.Conv2d(32 * 3 + D, 3, 3, 1, "same")

    def forward(self, cover_image, message):
        xs = []
        x = self.conv1(cover_image)
        xs.append(x)

        x = self.conv2(torch.cat(xs + [message], dim=1))
        xs.append(x)

        x = self.conv3(torch.cat(xs + [message], dim=1))
        # apply window attention + residual on the third feature map
        x = x + self.norm(self.attn(x))
        xs.append(x)

        x = self.conv4(torch.cat(xs + [message], dim=1))
        return cover_image + x

Decoder

In [ ]:
import torch
import torch.nn as nn

class AttentionDecoder(nn.Module):
    def __init__(self, D: int, window_size=8, num_heads=4):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 32, 3, 1, "same"),
            nn.LeakyReLU(inplace=True),
            nn.BatchNorm2d(32)
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 32, 3, 1, "same"),
            nn.LeakyReLU(inplace=True),
            nn.BatchNorm2d(32)
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(32 * 2, 32, 3, 1, "same"),
            nn.LeakyReLU(inplace=True),
            nn.BatchNorm2d(32)
        )

        self.attn = WindowAttention(32, window_size=window_size, num_heads=num_heads)
        self.norm = nn.BatchNorm2d(32)

        self.conv4 = nn.Conv2d(32 * 3, D, 3, 1, "same")

    def forward(self, stego_image):
        xs = []
        x = self.conv1(stego_image)
        xs.append(x)
        x = self.conv2(torch.cat(xs, dim=1))
        xs.append(x)
        x = self.conv3(torch.cat(xs, dim=1))
        # attention on the same scale as the encoder
        x = x + self.norm(self.attn(x))
        xs.append(x)
        return self.conv4(torch.cat(xs, dim=1))

Training

In [ ]:
from torch.nn.utils import clip_grad_norm_
import torch.nn.functional as F
import wandb
from tqdm import tqdm

class Trainer:
    """
    This training class jointly optimizes three networks:
    - Encoder: hides a binary message inside an image
    - Decoder: recovers the hidden message
    - Critic: distinguishes real vs generated images (Wasserstein GAN)

    For each batch:
    1. Sample a random binary message M ~ Ber(0.5)
    2. Update the critic using Wasserstein loss:
    Lc = C(real) - C(fake)
    3. Update encoder + decoder using:
    L = Ld + Ls + Lr
    where:
        Ld: decoding loss (binary cross entropy)
        Ls: similarity loss (MSE between cover and stego image)
        Lr: realism loss from critic

    After each epoch, evaluate using:
        RS-BPP (message capacity)
        PSNR (pixel-level distortion)
        SSIM (perceptual similarity)
    """
    def __init__(self, encoder, decoder, critic, D, device, sample_every=2):
        self.encoder = encoder.to(device)
        self.decoder = decoder.to(device)
        self.critic = critic.to(device)

        self.device = device
        self.D = D

        self.enc_dec_opt = torch.optim.Adam(
            list(self.encoder.parameters()) + list(self.decoder.parameters()),
            lr=1e-4,
        )
        self.critic_opt = torch.optim.Adam(self.critic.parameters(), lr=1e-4)

        self.grad_clip = 0.25
        self.critic_clip = 0.1
        self.sample_every = sample_every  # log image panel every N epochs

    def sample_message(self, N, H, W):
        """
        Inputs:
        - N: batch size
        - D: bits per pixel (data depth)
        - H, W: spatial dimensions
        - device: torch device (cpu or cuda)

        Returns:
        - message: N x D x H x W tensor of binary values {0,1}

        Description:
        - Generates a random binary message for each image in the batch
        - Each pixel stores D bits
        - Values are sampled from a Bernoulli(0.5) distribution
        """
        return torch.randint(0, 2, (N, self.D, H, W), device=self.device).float()

    def similarity_loss(self, cover, generated):
        """
        Inputs:
        - cover: N x 3 x H x W original image
        - generated: N x 3 x H x W stego image

        Returns:
        - scalar similarity loss

        Description:
        - Computes normalized mean squared error between cover and stego images
        - Matches paper formulation:
            Ls = (1 / (3 * H * W)) * ||cover - generated||^2
        - Encourages minimal visual distortion
        """
        _, _, H, W = cover.shape
        return ((cover - generated) ** 2).sum(dim=(1, 2, 3)).mean() / (3 * H * W)

    def train_epoch(self, loader, epoch):

        self.encoder.train()
        self.decoder.train()
        self.critic.train()

        total = {"Lc": 0.0, "Ld": 0.0, "Ls": 0.0, "Lr": 0.0, "L_total": 0.0, "Acc": 0.0}
        steps = 0

        tqdm_bar = tqdm(loader, desc=f"Epoch: {epoch}", leave=True)
        for cover in tqdm_bar:
            cover = cover.to(self.device)
            N, _, H, W = cover.shape

            # sample message to hide for this batch
            M = self.sample_message(N, H, W)

            # Critic
            with torch.no_grad():
                fake = self.encoder(cover, M)

            # Critic Update (Wasserstein GAN + Gradient clipping (stability) + Weight clipping to enforce Lipschitz constraint)
            real_score = self.critic(cover).mean()
            fake_score = self.critic(fake).mean()
            Lc = real_score - fake_score

            self.critic_opt.zero_grad()
            Lc.backward()
            clip_grad_norm_(self.critic.parameters(), self.grad_clip)
            self.critic_opt.step()

            for p in self.critic.parameters():
                p.data.clamp_(-self.critic_clip, self.critic_clip)

            # Encoder-Decoder Loss & Updates
            fake = self.encoder(cover, M)
            decoded = self.decoder(fake)

            # Ld (decoding loss): Binary cross entropy between decoded message and original message
            Ld = F.binary_cross_entropy_with_logits(decoded, M)
            # Ls (similarity loss): MSE between cover and stego image
            Ls = self.similarity_loss(cover, fake)
            # Lr (realness loss): Critic score of generated image
            Lr = self.critic(fake).mean()

            # L_total: combined enc-dec loss that actually drives gradients (paper Eq. 11); ablation: scale Ls by 100 to drive down Ls
            loss = Ld + 100.0 * Ls # + Lr

            self.enc_dec_opt.zero_grad()
            loss.backward()
            clip_grad_norm_(
                list(self.encoder.parameters()) + list(self.decoder.parameters()),
                self.grad_clip,
            )
            self.enc_dec_opt.step()

            with torch.no_grad():
                acc = ((decoded >= 0) == (M >= 0.5)).float().mean() # Measures fraction of correctly recovered bits

            total["Lc"] += Lc.item()
            total["Ld"] += Ld.item()
            total["Ls"] += Ls.item()
            total["Lr"] += Lr.item()
            total["L_total"] += loss.item()
            total["Acc"] += acc.item()
            steps += 1

            wandb.log({#"train/Lc": Lc.item(),
                       "train/Ld": Ld.item(),
                "train/Ls": Ls.item(), #"train/Lr": Lr.item(),
                "train/L_total": loss.item(), "train/Acc": acc.item()})
            tqdm_bar.set_postfix({
                "Lc": f"{Lc.item():.3f}",
                "Ld": f"{Ld.item():.3f}", "Acc": f"{acc.item():.3f}"})

        return {k: v / steps for k, v in total.items()}

    def train(self, train_loader, val_loader, epochs):
        # grab ONE fixed val batch up-front so the qualitative panel tracks
        # the same images across all epochs
        fixed_val_batch = next(iter(val_loader))

        for epoch in range(epochs):
            print(f"\nEpoch {epoch+1}")

            train_metrics = self.train_epoch(train_loader, epoch+1)

            print(
                f"Lc: {train_metrics['Lc']:.4f} | "
                f"Ld: {train_metrics['Ld']:.4f} | "
                f"Ls: {train_metrics['Ls']:.6f} | "
                f"Lr: {train_metrics['Lr']:.4f} | "
                f"L_total: {train_metrics['L_total']:.4f} | "
                f"Acc: {train_metrics['Acc']:.4f}"
            )

            # Evaluation metrics, see evaluate.py for details
            eval_metrics = evaluate(self.encoder, self.decoder, val_loader, self.D, device=self.device)

            print(
                f"RS-BPP: {eval_metrics['RS-BPP']:.4f} | "
                f"PSNR: {eval_metrics['PSNR']:.2f} | "
                f"SSIM: {eval_metrics['SSIM']:.4f} | "
                f"Acc: {eval_metrics['Acc']:.4f}"
            )

            # Build a single per-epoch log dict so eval scalars and qualitative
            # images share the same wandb _step (cleaner table view + 1 fewer
            # network call per epoch).
            log_dict = {
                "eval/RS-BPP": eval_metrics["RS-BPP"],
                "eval/PSNR":   eval_metrics["PSNR"],
                "eval/SSIM":   eval_metrics["SSIM"],
                "eval/Acc":    eval_metrics["Acc"],
                "epoch":       epoch + 1,
            }

            # Qualitative image panel on a fixed val batch, every N epochs
            # (also logs on the first epoch so we have a baseline)
            # _qualitative_samples_dict is defined in the Evaluation Functions cell
            if epoch == 0 or (epoch + 1) % self.sample_every == 0 or (epoch + 1) == epochs:
                log_dict.update(_qualitative_samples_dict(
                    self.encoder, self.decoder, fixed_val_batch,
                    D=self.D, device=self.device,
                ))

            wandb.log(log_dict)


Critic

In [ ]:
import torch
import torch.nn as nn

class Critic(nn.Module):
    """
    Spec: See Section 3.2.3 (Equation 7)
    Input Size: (N, 3, H, W)
    Output Size: (N, 1, H, W)
    """

    def __init__(self, hidden_dim=32):
        super(Critic, self).__init__()
        # (same as decoder)
        # conv_D-->D' blocks: (page 3, 4 in the paper)
        # (1) Conv2d with in_channel D, out_channel D', kernel size 3, stride 1, padding same (so padding = 1)
        # (2) LeakyRelU activation
        # (3) BatchNormalization
        # omit activation and batch norm if convolution block is last block in network

        self.hidden_dim = hidden_dim
        # cat operation: concat along the depth axis (the channel axis)
        self.a = nn.Conv2d(3, self.hidden_dim, 3, 1, 1) # Conv3->32
        self.b = nn.Conv2d(self.hidden_dim, self.hidden_dim, 3, 1, 1) # Conv32->32
        self.c = nn.Conv2d(self.hidden_dim, self.hidden_dim, 3, 1, 1) # Conv32->32
        self.d = nn.Conv2d(self.hidden_dim, 1, 3, 1, 1) # Conv32->1

        self.leakyrelu = nn.LeakyReLU(inplace=True)
        self.batchnorm = nn.BatchNorm2d(self.hidden_dim)


    def forward(self, x):
        a_x = self.batchnorm(self.leakyrelu(self.a(x)))
        b_x = self.batchnorm(self.leakyrelu(self.b(a_x)))
        c_x = self.batchnorm(self.leakyrelu(self.c(b_x)))
        d_x = self.d(c_x) # N x 1 x H x W
        d_x = d_x.mean(dim=[2, 3]).squeeze(1) # find single mean for each channel (mean across H x W dimension --> N x 1 --> (N, ) as a result of squeeze)

        return d_x # dimension: (N, )

Google Drive mounting for checkpoints

In [ ]:
SAVE_DIR = "/content/drive/MyDrive/CS4782-Final-Project/steganogan_checkpoints/no_critic"
os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
print(SAVE_DIR)
print(os.path.exists(os.path.join(SAVE_DIR, "basic_D1.pt")))

/content/drive/MyDrive/CS4782-Final-Project/steganogan_checkpoints/no_critic
True


Configs

In [ ]:
# Config sweep

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Paper trains: 3 architectures x 6 data depths = 18 configs
# lr=1e-4, grad_clip=0.25, critic_clip=0.1, epochs=32 (all fixed in paper)
EPOCHS   = 32
LR       = 1e-4

encoder_variants = {
    # "basic":    BasicEncoder,
    # "residual": ResidualEncoder,
    #"dense":    DenseEncoder,
    "attention": AttentionEncoder,
}

configs = [
    {"arch": arch_name, "D": D, "label": f"{arch_name}_D{D}"}
    for arch_name in ["attention"]
    for D in [6]#, 3, 6]
]

all_results = {}

for cfg in configs:

    checkpoint_path = os.path.join(SAVE_DIR, f"{cfg['label']}.pt")

    # skip + load saved metrics
    if os.path.exists(checkpoint_path):
        print(f"Skipping {cfg['label']} (already trained)")

        ckpt = torch.load(checkpoint_path, map_location=device)

        all_results[cfg["label"]] = {
            **ckpt["metrics"],
            "arch": cfg["arch"],
            "D": cfg["D"],
        }

        continue

    print(f"\n{'='*60}")
    print(f"Running config: {cfg['label']}  (arch={cfg['arch']}, D={cfg['D']})")
    print('='*60)

    # init wandb FIRST so we fail fast (auth/network) before allocating GPU memory.
    wandb.init(
        project="steganogan_metrics",
        name=cfg["label"],
        config={"arch": cfg["arch"], "D": cfg["D"], "epochs": EPOCHS, "lr": LR},
        reinit=True,
    )

    wandb.define_metric("epoch")
    wandb.define_metric("eval/*", step_metric="epoch")
    wandb.define_metric("val/*",  step_metric="epoch")

    EncoderClass = encoder_variants[cfg["arch"]]

    encoder = EncoderClass(D=cfg["D"])
    decoder = AttentionDecoder(D=cfg["D"])
    critic  = Critic()

    trainer = Trainer(encoder, decoder, critic, D=cfg["D"], device=device)

    trainer.train(train_loader, valid_loader, epochs=EPOCHS)

    final = evaluate(encoder, decoder, valid_loader, cfg["D"], device=device)

    all_results[cfg["label"]] = {
        **final,
        "arch": cfg["arch"],
        "D": cfg["D"]
    }

    torch.save({
        "encoder_state_dict": encoder.state_dict(),
        "decoder_state_dict": decoder.state_dict(),
        "critic_state_dict": critic.state_dict(),
        "config": cfg,
        "metrics": final,
    }, checkpoint_path)

    print(f"Saved checkpoint to {checkpoint_path}")

    print(
        f"\n[{cfg['label']}] RS-BPP: {final['RS-BPP']:.4f} | "
        f"PSNR: {final['PSNR']:.2f} dB | "
        f"SSIM: {final['SSIM']:.4f} | "
        f"Acc: {final['Acc']:.4f}"
    )

    wandb.finish()

# Summary table (based on paper Table 1)
print(f"\n{'='*72}")
print(f"{'D':<4} {'Arch':<12} {'RS-BPP':>8} {'PSNR':>8} {'SSIM':>8} {'Acc':>8}")
print('-'*72)

for D in [1, 3, 6]:
    for arch in ["attention"]:

        label = f"{arch}_D{D}"

        if label not in all_results:
            print(f"D={D}  {arch:<12} MISSING")
            continue

        m = all_results[label]

        print(
            f"D={D}  {arch:<12} "
            f"{m['RS-BPP']:>8.4f} "
            f"{m['PSNR']:>8.2f} "
            f"{m['SSIM']:>8.4f} "
            f"{m['Acc']:>8.4f}"
        )

    print()


Running config: attention_D6  (arch=attention, D=6)


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.



Epoch 1


Epoch: 1: 100%|██████████| 200/200 [02:42<00:00,  1.23it/s, Ld=0.648, Acc=0.629]


Lc: 0.0000 | Ld: 0.6777 | Ls: 0.051734 | Lr: 0.0000 | L_total: 5.8510 | Acc: 0.5818
RS-BPP: 1.3275 | PSNR: 26.08 | SSIM: 0.7524 | Acc: 0.6106

Epoch 2


...

Lc: 0.0000 | Ld: 0.4545 | Ls: 0.000495 | Lr: 0.0000 | L_total: 0.5040 | Acc: 0.7012
RS-BPP: 2.3530 | PSNR: 39.41 | SSIM: 0.9645 | Acc: 0.6961

Epoch 32


Epoch: 32: 100%|██████████| 200/200 [02:44<00:00,  1.21it/s, Ld=0.434, Acc=0.711]


Lc: 0.0000 | Ld: 0.4553 | Ls: 0.000496 | Lr: 0.0000 | L_total: 0.5050 | Acc: 0.7010
RS-BPP: 2.3654 | PSNR: 38.88 | SSIM: 0.9619 | Acc: 0.6971
Saved checkpoint to /content/drive/MyDrive/CS4782-Final-Project/steganogan_checkpoints/no_critic/attention_D6.pt

[attention_D6] RS-BPP: 2.3656 | PSNR: 38.88 dB | SSIM: 0.9619 | Acc: 0.6971


epoch,▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇███
eval/Acc,▁▂▄▆▇▇▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇████████
eval/PSNR,▁▄▅▆▆▇▇▇▇▇▇▇▇▇███▇██████████████
eval/RS-BPP,▁▂▄▆▇▇▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇████████
eval/SSIM,▁▆▇▇▇█▇█████████████████████████
train/Acc,▁▂▁▃▃▆▆▅▇▆▆▅▇▆▃▄▅▅▄▆▄▅▆▆▆▆▆▆▆▇▇▆▇▆▇▆▇█▇▇
train/L_total,█▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/Ld,█▇▇▆▆▆▆▆▅▄▄▄▄▃▂▃▂▄▃▄▂▂▃▃▁▂▂▂▂▃▂▂▂▁▃▂▁▂▄▃
train/Ls,█▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,32
eval/Acc,0.69712



D    Arch           RS-BPP     PSNR     SSIM      Acc
------------------------------------------------------------------------
D=1  attention    MISSING

D=3  attention    MISSING

D=6  attention      2.3656    38.88   0.9619   0.6971



In [ ]:
from google.colab import runtime
runtime.unassign()